In [2]:
from pathlib import Path
from getpass import getpass
from concurrent.futures import ThreadPoolExecutor, as_completed

import base64
import mimetypes
import re
import sys
import time

import pandas as pd
import requests

sys.path.append(str(Path("../..").resolve()))
from image_prep import image_preprocessing

API_KEY = getpass("OpenRouter API key: ")

DATASET_FOLDER = Path("../ImageDataset")
OUTPUT_FOLDER = Path("PreprocessingModelResults")

OUTPUT_FOLDER.mkdir(
    parents=True,
    exist_ok=True,
)


MODELS = {
    # OpenAI
    "gpt_5_6_sol": "openai/gpt-5.6-sol",
    "gpt_5_6_terra": "openai/gpt-5.6-terra",
    "gpt_5_6_luna": "openai/gpt-5.6-luna",
    "gpt_5_5": "openai/gpt-5.5",
    "gpt_5_4_mini": "openai/gpt-5.4-mini",

    # Anthropic
    "claude_sonnet_5": "anthropic/claude-sonnet-5",
    "claude_fable_5": "anthropic/claude-fable-5",
    "claude_opus_4_8": "anthropic/claude-opus-4.8",
    "claude_haiku_4_5": "anthropic/claude-haiku-4.5",

    # Google
    "gemini_3_5_flash": "google/gemini-3.5-flash",
    "gemini_3_1_pro": "google/gemini-3.1-pro-preview",
    "gemini_3_1_flash_lite": "google/gemini-3.1-flash-lite",
    "gemini_3_flash": "google/gemini-3-flash-preview",

    # Qwen
    "qwen_3_7_plus": "qwen/qwen3.7-plus",
    "qwen_3_6_flash": "qwen/qwen3.6-flash",
    "qwen_3_6_35b": "qwen/qwen3.6-35b-a3b",
    "qwen_3_5_397b": "qwen/qwen3.5-397b-a17b",
    "qwen_3_5_122b": "qwen/qwen3.5-122b-a10b",
}


PROMPT = (
    "Return the answer using exactly two XML tags: "
    "<heading> and <description>. "
    "Inside <heading>, write a short and specific title of 2-6 words that clearly identifies what is visible in the image. "
    "Inside <description>, identify the main subject and describe it in one concise, factual paragraph of 2-3 sentences. "
    "Include distinctive colors, materials, shapes, context, and any readable text that would help a visual search engine find similar objects or scenes. "
    "Avoid decorative language, unsupported assumptions, and phrases such as 'The image shows.' "
    "Express uncertain identifications cautiously using words such as 'likely,' 'possibly,' or 'appears to be.' "
    "Use this exact output format: "
    "<heading>Specific image title</heading>"
    "<description>Concise factual description.</description>"
)


ACCEPTED_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
}

# Maximum number of simultaneous OpenRouter requests
MAX_WORKERS = 8

def encode_image(
    image_path,
    use_preprocessing,
):
    image_bytes = image_path.read_bytes()

    mime_type = (
        mimetypes.guess_type(
            image_path.name
        )[0]
        or "image/jpeg"
    )

    if use_preprocessing:
        image_bytes = image_preprocessing(
            image_bytes
        )

        mime_type = "image/jpeg"

    image_b64 = base64.b64encode(
        image_bytes
    ).decode("utf-8")

    return mime_type, image_b64


def describe_image(
    image_b64,
    mime_type,
    model_name,
):
    payload = {
        "model": model_name,
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": PROMPT,
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": (
                                f"data:{mime_type};base64,"
                                + image_b64
                            )
                        },
                    },
                ],
            }
        ],
        "temperature": 0,
        "max_tokens": 350,
    }

    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": (
                f"Bearer {API_KEY}"
            ),
            "Content-Type": (
                "application/json"
            ),
        },
        json=payload,
        timeout=180,
    )

    response.raise_for_status()

    response_data = response.json()

    return response_data[
        "choices"
    ][0]["message"]["content"]


def extract_xml(response_text):
    heading_match = re.search(
        r"<heading>(.*?)</heading>",
        response_text,
        re.DOTALL | re.IGNORECASE,
    )

    description_match = re.search(
        r"<description>(.*?)</description>",
        response_text,
        re.DOTALL | re.IGNORECASE,
    )

    heading = (
        heading_match.group(1).strip()
        if heading_match
        else ""
    )

    description = (
        description_match.group(1).strip()
        if description_match
        else response_text.strip()
    )

    format_ok = bool(
        heading_match
        and description_match
    )

    return (
        heading,
        description,
        format_ok,
    )


if not DATASET_FOLDER.exists():
    raise FileNotFoundError(
        "Dataset folder was not found: "
        f"{DATASET_FOLDER.resolve()}"
    )


image_paths = sorted(
    path
    for path in DATASET_FOLDER.iterdir()
    if path.is_file()
    and path.suffix.lower()
    in ACCEPTED_EXTENSIONS
)


if not image_paths:
    raise ValueError(
        "No supported images were found in "
        f"{DATASET_FOLDER.resolve()}"
    )


print(
    f"Found {len(image_paths)} images."
)

print(
    f"Models to evaluate: {len(MODELS)}"
)


encoded_images = {
    "before": {},
    "after": {},
}


for image_path in image_paths:
    try:
        encoded_images[
            "before"
        ][image_path] = encode_image(
            image_path,
            use_preprocessing=False,
        )

        encoded_images[
            "after"
        ][image_path] = encode_image(
            image_path,
            use_preprocessing=True,
        )

    except Exception as error:
        encoded_images[
            "before"
        ][image_path] = error

        encoded_images[
            "after"
        ][image_path] = error


for model_index, (
    model_label,
    model_name,
) in enumerate(
    MODELS.items(),
    start=1,
):
    for variant in [
        "before",
        "after",
    ]:
        output_file = OUTPUT_FOLDER / (
            f"submission_{model_label}_{variant}.csv"
        )

        print()
        print("=" * 70)

        print(
            f"Model {model_index}/{len(MODELS)}: "
            f"{model_name}"
        )

        print(
            f"Variant: {variant} "
            "image preprocessing"
        )

        print(
            f"Output: {output_file}"
        )

        print("=" * 70)

        results = []

        def process_image(image_path):
            encoded_value = encoded_images[
                variant
            ][image_path]

            start_time = (
                time.perf_counter()
            )

            try:
                if isinstance(
                    encoded_value,
                    Exception,
                ):
                    raise encoded_value

                (
                    mime_type,
                    image_b64,
                ) = encoded_value

                response_text = describe_image(
                    image_b64,
                    mime_type,
                    model_name,
                )

                (
                    heading,
                    description,
                    format_ok,
                ) = extract_xml(
                    response_text
                )

                status = "success"

            except Exception as error:
                response_text = (
                    f"ERROR: {error}"
                )

                heading = ""
                description = ""
                format_ok = False
                status = "error"

            elapsed_seconds = round(
                time.perf_counter()
                - start_time,
                2,
            )

            return {
                "image_path": (
                    image_path.as_posix()
                ),
                "model": model_name,
                "variant": variant,
                "status": status,
                "response_time_seconds": (
                    elapsed_seconds
                ),
                "format_ok": format_ok,
                "heading": heading,
                "description": description,
                "ai_response": response_text,
            }

        worker_count = min(
            MAX_WORKERS,
            len(image_paths),
        )

        with ThreadPoolExecutor(
            max_workers=worker_count
        ) as executor:
            future_to_image = {
                executor.submit(
                    process_image,
                    image_path,
                ): image_path
                for image_path in image_paths
            }

            for (
                completed_index,
                future,
            ) in enumerate(
                as_completed(
                    future_to_image
                ),
                start=1,
            ):
                image_path = (
                    future_to_image[
                        future
                    ]
                )

                result = future.result()

                results.append(
                    result
                )

                print(
                    f"[{completed_index}/"
                    f"{len(image_paths)}] "
                    f"Completed: "
                    f"{image_path.name} - "
                    f"{result['status']}"
                )

                ordered_results = sorted(
                    results,
                    key=lambda item: (
                        image_paths.index(
                            Path(
                                item[
                                    "image_path"
                                ]
                            )
                        )
                    ),
                )

                pd.DataFrame(
                    ordered_results
                ).to_csv(
                    output_file,
                    index=False,
                    encoding="utf-8-sig",
                )

        results = sorted(
            results,
            key=lambda item: (
                image_paths.index(
                    Path(
                        item["image_path"]
                    )
                )
            ),
        )

        successful_requests = sum(
            result["status"]
            == "success"
            for result in results
        )

        print(
            f"Completed {model_name} - "
            f"{variant}: "
            f"{successful_requests}/"
            f"{len(results)} "
            "successful requests."
        )


print()
print("Evaluation completed.")

print(
    "Results were saved in: "
    f"{OUTPUT_FOLDER.resolve()}"
)

Found 10 images.
Models to evaluate: 18

Model 1/18: openai/gpt-5.6-sol
Variant: before image preprocessing
Output: PreprocessingModelResults\submission_gpt_5_6_sol_before.csv
[1/10] Completed: Blurry_city.jpg - error
[2/10] Completed: dark_light_man.jpg - error
[3/10] Completed: night_traffic_blur.jpg - error
[4/10] Completed: insect.jpg - error
[5/10] Completed: cyclist.jpg - error
[6/10] Completed: crowded_market.jpg - error
[7/10] Completed: rain_window_reflection.jpg - error
[8/10] Completed: distant_road_sign.jpg - error
[9/10] Completed: camouflaged_animal.jpg - error
[10/10] Completed: smoke_ships.jpg - error
Completed openai/gpt-5.6-sol - before: 0/10 successful requests.

Model 1/18: openai/gpt-5.6-sol
Variant: after image preprocessing
Output: PreprocessingModelResults\submission_gpt_5_6_sol_after.csv


KeyboardInterrupt: 